In [ ]:
# ===========================================================================
# CELL 1 - IMPORTS AND CONFIGURATION
# ===========================================================================

import re
import time
import json
import math
import random
import requests
from datetime import datetime
from pathlib import Path
from typing import Optional

import pandas as pd

# ---------------------------------------------------------------------------
# USER CONFIGURATION - edit these before running
# ---------------------------------------------------------------------------

# Directory where all output CSVs will be saved
OUTPUT_DIR = Path(r"C:\Users\danie\OneDrive\Python\Ciaran\myhomes")

# Base URL for the JSON feed
FEED_BASE_URL = "https://www.myhome.ie/feed/residential/ireland/new-homes/property-for-sale"

# Seconds to pause between page requests
REQUEST_DELAY = 1.5

# Set to an integer to limit pages fetched (for testing). None = fetch all.
PAGE_LIMIT = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Browser-like headers so the server does not reject the request
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.myhome.ie/",
}

print("Configuration loaded.")
print(f"Feed URL : {FEED_BASE_URL}")
print(f"Output   : {OUTPUT_DIR}")

In [ ]:
# ===========================================================================
# CELL 2 - COUNTY / PROVINCE LOOKUPS AND UNIQUE ID GENERATOR
# ===========================================================================
# Generates IDs in the format MH{COUNTY_CODE}{YEAR}{SEQUENCE}
# e.g. MHMH2026001 (Meath, 2026, 1st listing)
#      MHD12026042 (Dublin, 2026, 42nd listing)
#
# The 'MH' prefix distinguishes MyHome IDs from daft.ie IDs if the
# two datasets are ever merged into one database.
# ===========================================================================

COUNTY_CODES = {
    "Dublin":    "D1", "Cork":      "CK", "Galway":    "GA",
    "Limerick":  "LK", "Waterford": "WD", "Tipperary": "TP",
    "Clare":     "CE", "Kerry":     "KY", "Kildare":   "KE",
    "Meath":     "MH", "Louth":     "LH", "Wexford":   "WX",
    "Wicklow":   "WW", "Kilkenny":  "KK", "Laois":     "LS",
    "Offaly":    "OY", "Westmeath": "WH", "Carlow":    "CW",
    "Longford":  "LD", "Mayo":      "MO", "Sligo":     "SO",
    "Roscommon": "RN", "Leitrim":   "LM", "Donegal":   "DL",
    "Monaghan":  "MN", "Cavan":     "CN", "Antrim":    "AM",
}

PROVINCE_MAP = {
    "Dublin":"Leinster",   "Kildare":"Leinster",  "Wicklow":"Leinster",
    "Wexford":"Leinster",  "Kilkenny":"Leinster",  "Laois":"Leinster",
    "Offaly":"Leinster",   "Meath":"Leinster",     "Westmeath":"Leinster",
    "Carlow":"Leinster",   "Louth":"Leinster",     "Longford":"Leinster",
    "Cork":"Munster",      "Waterford":"Munster",  "Limerick":"Munster",
    "Kerry":"Munster",     "Tipperary":"Munster",  "Clare":"Munster",
    "Galway":"Connacht",   "Mayo":"Connacht",      "Sligo":"Connacht",
    "Roscommon":"Connacht","Leitrim":"Connacht",
    "Donegal":"Ulster",    "Monaghan":"Ulster",    "Cavan":"Ulster",
    "Antrim":"Ulster",
}

NAMES = {
    "CORK":"Cork", "KILDARE":"Kildare", "WEXFORD":"Wexford", "WICKLOW":"Wicklow",
    "WESTMEATH":"Westmeath", "MEATH":"Meath", "GALWAY":"Galway", "LOUTH":"Louth",
    "WATERFORD":"Waterford", "LIMERICK":"Limerick", "KILKENNY":"Kilkenny",
    "LAOIS":"Laois", "OFFALY":"Offaly", "SLIGO":"Sligo", "CLARE":"Clare",
    "TIPPERARY":"Tipperary", "KERRY":"Kerry", "CAVAN":"Cavan", "MAYO":"Mayo",
    "LEITRIM":"Leitrim", "ANTRIM":"Antrim", "DONEGAL":"Donegal",
    "MONAGHAN":"Monaghan", "ROSCOMMON":"Roscommon", "CARLOW":"Carlow",
    "LONGFORD":"Longford", "DUBLIN":"Dublin",
}

DUBLIN_SUBURBS = {
    "SANDYMOUNT", "STILLORGAN", "CLONTARF", "RANELAGH", "RATHMINES",
    "BALLSBRIDGE", "BLACKROCK", "DUNDRUM", "CABINTEELY", "FOXROCK",
    "GLASNEVIN", "LUCAN", "MALAHIDE", "SWORDS", "CLONDALKIN",
    "TALLAGHT", "ADAMSTOWN", "LEOPARDSTOWN", "MOUNT MERRION",
    "CHERRYWOOD", "KILMACUD", "DONNYBROOK",
}


def standardise_county(x) -> Optional[str]:
    if not x or pd.isna(x):
        return None
    s = re.sub(r"\s+", " ", str(x).strip())
    # Strip trailing dot e.g. "Kildare." -> "Kildare"
    s = s.rstrip(".")
    # Remove City/Town suffixes
    s = re.sub(r"\s+(City Centre|City Center|City|Town)\s*$", "", s, flags=re.IGNORECASE).strip()
    # Handle both "Co.Dublin" (no space) and "Co. Dublin" (with space)
    s = re.sub(r"^Co\.?\s*", "", s, flags=re.IGNORECASE).strip()
    # Dublin number variants e.g. "Dublin 15"
    if re.match(r"^Dublin\s*\d*$", s, re.IGNORECASE):
        return "Dublin"
    # Garbled all-uppercase fragments e.g. "LMBALLIN", "SOBMOTE" - not a county
    if re.match(r"^[A-Z]{4,}$", s):
        return None
    # Direct county name match
    result = NAMES.get(s.upper())
    if result:
        return result
    # Dublin suburbs that appear in addresses instead of the county name
    if s.upper() in DUBLIN_SUBURBS:
        return "Dublin"
    # Cork sub-regions e.g. "East Cork", "West Cork"
    if "CORK" in s.upper():
        return "Cork"
    return s.title()


def generate_unique_id(county: Optional[str], year: int, sequence: int) -> str:
    """
    Builds an ID like 'MHWW2026001' (MyHome, Wicklow, 2026, 1st listing).
    """
    code = COUNTY_CODES.get(county, "XX") if county else "XX"
    return f"MH{code}{year}{sequence:03d}"


def assign_unique_ids(df: pd.DataFrame, year: int) -> pd.Series:
    """
    Assigns a unique ID to every row.
    Rows in the same county are numbered sequentially from 001.
    """
    ids = []
    counters = {}
    for county in df["county_standard"]:
        counters[county] = counters.get(county, 0) + 1
        ids.append(generate_unique_id(county, year, counters[county]))
    return pd.Series(ids, index=df.index)


print("Lookups and ID generator ready.")
print("Example IDs:")
for county in ["Wicklow", "Dublin", "Meath", None]:
    print(f"  {str(county or 'None'):12} -> {generate_unique_id(county, 2026, 1)}")

In [ ]:
# ===========================================================================
# CELL 3 - PRICE PARSER
# ===========================================================================
# MyHome stores prices in a Value field as a plain integer string.
# For price-range listings the Value field concatenates both numbers:
#   e.g. '€405,000 to €520,000' -> Value = '405000520000'
#
# We detect ranges by checking if the Value string is longer than a
# plausible single price (max 8 digits = EUR 99,999,999).
# If longer we split it in half into price_min and price_max.
# Single prices are stored in both columns for consistency.
# POA listings have an empty Value - both columns will be None.
#
# "from €X" listings where X is below 10,000 are not sale prices -
# they are deposits or weekly rents. These are treated as POA.
# "from €X" where X is >= 10,000 is a valid minimum sale price.
# ===========================================================================

MAX_SINGLE_PRICE_DIGITS = 8


def parse_price(display: str, value: str) -> tuple:
    """
    Returns (price_min, price_max, price_display, is_poa).
    price_min and price_max are integers or None.
    price_display is the raw string e.g. '€405,000 to €520,000'.
    is_poa is True when no price is available.
    """
    display = (display or "").strip()
    value   = (value   or "").strip()

    # "from €X" handling - check if the amount is a real sale price
    if display.lower().startswith("from"):
        digits = re.sub(r"\D", "", value)
        if digits and int(digits) < 10_000:
            # Too low to be a sale price - treat as POA
            return None, None, "POA", True

    is_poa = (display.upper() == "POA" or value == "")

    if is_poa:
        return None, None, display or "POA", True

    # Strip everything except digits
    digits = re.sub(r"\D", "", value)

    if len(digits) <= MAX_SINGLE_PRICE_DIGITS:
        # Single price - store in both min and max
        p = int(digits)
        return p, p, display, False
    else:
        # Range - split in half
        mid   = len(digits) // 2
        p_min = int(digits[:mid])
        p_max = int(digits[mid:])
        # Sanity check: min should be less than max
        if p_min >= p_max:
            p = int(digits)
            return p, p, display, False
        return p_min, p_max, display, False


# Tests against the real data from the JSON feed
print("Price parser tests:")
test_cases = [
    ("€405,000 to €520,000", "405000520000"),
    ("€720,000",             "720000"),
    ("POA",                  ""),
    ("€1,250,000",           "1250000"),
    ("€550,000 to €715,000", "550000715000"),
    ("from €370,000",        "370000"),    # valid minimum price
    ("from €3,306",          "3306"),      # deposit/rent - should become POA
    ("from €800",            "800"),       # deposit/rent - should become POA
]
for disp, val in test_cases:
    mn, mx, d, poa = parse_price(disp, val)
    print(f"  {disp:30} -> min={mn}  max={mx}  poa={poa}")

In [ ]:
# ===========================================================================
# CELL 4 - SINGLE LISTING PARSER
# ===========================================================================
# Converts one JSON property object into a flat dictionary ready for
# a pandas row. Called once per listing in Cell 6.
#
# All fields come directly from the JSON - no browser or HTML parsing
# needed. This is why myhome.ie is simpler than daft.ie.
# ===========================================================================

def parse_listing(prop: dict) -> dict:
    """
    Extracts and cleans all available fields from one MyHome property dict.

    Parameters
    ----------
    prop : dict
        One element from the 'Properties' list in the JSON feed.

    Returns
    -------
    Flat dict with standardised field names.
    """

    # --- ID and URL ---
    myhome_id = str(prop.get("PropertyId", ""))
    url = (prop.get("Url") or "").rstrip("}")

    # --- Address ---
    address_obj  = prop.get("Address") or {}
    full_address = (address_obj.get("FullAddress") or "").strip()

    # County is the last comma-separated segment of the full address
    # e.g. 'Moy Road, Summerhill, Meath' -> 'Meath'
    county_raw = full_address.split(",")[-1].strip() if "," in full_address else ""

    # --- Property Details ---
    details   = prop.get("PropertyDetails") or {}
    prop_type = (details.get("Type") or "").strip() or None

    # Beds: '3 beds' -> 3
    beds_raw   = details.get("Beds") or ""
    beds_match = re.search(r"(\d+)", beds_raw)
    beds       = int(beds_match.group(1)) if beds_match else None

    # Baths: '2 baths' -> 2 (not always present in the feed)
    baths_raw   = details.get("Baths") or ""
    baths_match = re.search(r"(\d+)", baths_raw)
    baths       = int(baths_match.group(1)) if baths_match else None

    # Area: only accept values above 0
    # The feed returns 0 when area is not provided - we treat 0 as missing
    area_raw = details.get("FloorAreaSqM")
    if area_raw is not None and float(area_raw) > 0:
        area_m2 = float(area_raw)
    else:
        area_m2 = None

    # --- Price ---
    price_obj = prop.get("Price") or {}
    price_min, price_max, price_display, is_poa = parse_price(
        price_obj.get("Display", ""),
        price_obj.get("Value",   "")
    )

    # Mid-point price - useful for sorting and analysis when only a range
    # is known rather than an exact figure
    price_mid = None
    if price_min is not None and price_max is not None:
        price_mid = (price_min + price_max) // 2

    # Price per m2 using mid-point price
    price_per_m2 = None
    if price_mid is not None and area_m2 and area_m2 > 0:
        price_per_m2 = round(price_mid / area_m2)

    # --- Agent ---
    agent_obj  = prop.get("Agent") or {}
    agent_name = (agent_obj.get("Name") or "").strip() or None

    # --- Media ---
    media_obj  = prop.get("Media") or {}
    main_photo = media_obj.get("MainPhoto") or None

    # --- Status ---
    listing_obj = prop.get("Listing") or {}
    status      = listing_obj.get("Status") or None

    return {
        "myhome_id"     : myhome_id,
        "url"           : url,
        "full_address"  : full_address,
        "county_raw"    : county_raw,
        "property_type" : prop_type,
        "beds"          : beds,
        "baths"         : baths,
        "area_m2"       : area_m2,
        "price_display" : price_display,
        "price_min"     : price_min,
        "price_max"     : price_max,
        "price_mid"     : price_mid,
        "price_per_m2"  : price_per_m2,
        "is_poa"        : is_poa,
        "agent_name"    : agent_name,
        "main_photo_url": main_photo,
        "status"        : status,
    }


print("parse_listing function loaded.")

In [ ]:
# ===========================================================================
# CELL 5 - FETCH ALL PAGES USING SELENIUM
# ===========================================================================
# The myhome.ie JSON feed blocks plain HTTP requests.
# We use Selenium to open the feed URL in a real browser, which passes
# all anti-bot checks, then read the page source which contains the JSON.
#
# Sometimes the browser wraps the JSON in HTML tags or adds extra content.
# We extract just the JSON object by finding the first { and last } in
# whatever the browser returns, ignoring anything around it.
# ===========================================================================

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
options.add_argument("--window-size=1400,900")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

all_properties = []


def fetch_page_json(page_num: int) -> dict:
    """
    Navigates to a feed page and extracts the JSON from the page source.
    Handles cases where the browser wraps the JSON in HTML tags or adds
    extra content around it.
    """
    url = f"{FEED_BASE_URL}?page={page_num}" if page_num > 1 else FEED_BASE_URL
    driver.get(url)
    time.sleep(3)

    # Get the raw text from the page body
    raw = driver.find_element("tag name", "body").text

    # Find the first { and last } to extract just the JSON object,
    # ignoring any HTML or extra content the browser may have added
    start = raw.find("{")
    end   = raw.rfind("}") + 1
    raw   = raw[start:end]

    return json.loads(raw)


# --- Page 1: fetch and discover total pages ---
print("Fetching page 1...")
data_p1 = fetch_page_json(1)

paging        = data_p1.get("Paging", {})
total_results = paging.get("TotalResults", 0)
page_size     = paging.get("PageSize", 20)
total_pages   = math.ceil(total_results / page_size)

if PAGE_LIMIT:
    total_pages = min(total_pages, PAGE_LIMIT)

print(f"Total listings : {total_results}")
print(f"Page size      : {page_size}")
print(f"Total pages    : {total_pages}")

props_p1 = data_p1.get("Properties", [])
all_properties.extend(props_p1)
print(f"Page  1/{total_pages} -> {len(props_p1)} listings collected")

# --- Pages 2 onwards ---
for page_num in range(2, total_pages + 1):
    try:
        time.sleep(REQUEST_DELAY + random.uniform(0, 0.5))
        data  = fetch_page_json(page_num)
        props = data.get("Properties", [])
        all_properties.extend(props)
        print(f"Page {page_num:2d}/{total_pages} -> {len(props)} listings collected")
    except Exception as e:
        print(f"Page {page_num:2d}/{total_pages} -> ERROR: {e}")

driver.quit()
print(f"\nTotal listings fetched: {len(all_properties)}")

In [ ]:
# ===========================================================================
# CELL 6 - PARSE ALL LISTINGS INTO A DATAFRAME
# ===========================================================================
# Calls parse_listing() on every property dict collected in Cell 5,
# builds a DataFrame, enforces correct column types, and removes any
# duplicate listings that appeared across multiple pages.
# ===========================================================================

rows = [parse_listing(p) for p in all_properties]
df   = pd.DataFrame(rows)

# Record when this data was fetched
df["date_scraped"] = datetime.now().strftime("%Y-%m-%d")

# Enforce correct types
# Int64 (capital I) is pandas nullable integer - preserves the difference
# between 0 and missing, unlike float which converts None to NaN
df["date_scraped"] = pd.to_datetime(df["date_scraped"])

# beds and baths are stored as integers not floats e.g. 4 not 4.0
for int_col in ["beds", "baths", "price_min", "price_max", "price_mid", "price_per_m2"]:
    df[int_col] = pd.to_numeric(df[int_col], errors="coerce").astype("Int64")

# area_m2 stays as float to preserve decimal precision e.g. 112.97
df["area_m2"] = pd.to_numeric(df["area_m2"], errors="coerce")
df["is_poa"]  = df["is_poa"].astype(bool)

# Remove duplicate listings - same myhome_id can appear on multiple pages
before = len(df)
df = df.drop_duplicates(subset="myhome_id").reset_index(drop=True)
after  = len(df)
if before != after:
    print(f"Removed {before - after} duplicate listings.")

print(f"DataFrame shape: {df.shape}")
print("\nColumn non-null counts:")
print(df.notna().sum().to_string())
print("\nFirst record:")
print(df.iloc[0].to_string())

In [ ]:
# ===========================================================================
# CELL 7 - COUNTY, PROVINCE AND UNIQUE ID
# ===========================================================================
# Tags every row with a standardised county name, province, and unique ID.
# Sorts the DataFrame geographically before assigning IDs so sequences
# are consistent across runs.
# ===========================================================================

df["county_standard"] = df["county_raw"].apply(standardise_county)
df["province"]        = df["county_standard"].map(PROVINCE_MAP).fillna("Unknown")

# Sort before assigning IDs so same county always gets same sequence
df = df.sort_values(
    by=["province", "county_standard", "full_address"],
    ascending=True,
    na_position="last"
).reset_index(drop=True)

scrape_year     = datetime.now().year
df["unique_id"] = assign_unique_ids(df, scrape_year)

# Reorder columns - identifiers first, then address, then price, then details
COL_ORDER = [
    "unique_id", "myhome_id", "full_address", "county_standard", "province",
    "date_scraped", "property_type", "beds", "baths", "area_m2",
    "price_display", "price_min", "price_max", "price_mid", "price_per_m2",
    "is_poa", "agent_name", "status", "url", "main_photo_url",
]
for c in COL_ORDER:
    if c not in df.columns:
        df[c] = pd.NA
df = df[COL_ORDER]

print("County / province / unique ID assigned.")
print("\nProvince breakdown:")
print(
    df.groupby("province", dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .to_string(index=False)
)
print("\nSample IDs:")
print(df[["unique_id", "myhome_id", "county_standard", "full_address"]].head(8).to_string(index=False))

In [ ]:
# ===========================================================================
# CELL 8 - DATA QUALITY SENSE-CHECK
# ===========================================================================
# Flags rows with suspicious values. Rows are NOT removed - they are
# flagged for manual review and exported to a separate CSV in Cell 9.
# ===========================================================================

print("DATA QUALITY REPORT")
print("=" * 60)

flags = {}

# --- Area ---
# Average Irish new home is ~100 m2. Flag below 20 or above 500.
bad_area_low  = df[df["area_m2"].notna() & (df["area_m2"] < 20)]
bad_area_high = df[df["area_m2"].notna() & (df["area_m2"] > 500)]
if len(bad_area_low):
    print(f"AREA - {len(bad_area_low)} row(s) with area_m2 < 20 (implausibly small):")
    print(bad_area_low[["unique_id", "full_address", "area_m2"]].to_string())
    flags.update({i: "area_too_small" for i in bad_area_low.index})
if len(bad_area_high):
    print(f"AREA - {len(bad_area_high)} row(s) with area_m2 > 500 (unusually large - check):")
    print(bad_area_high[["unique_id", "full_address", "area_m2"]].to_string())
    flags.update({i: "area_too_large" for i in bad_area_high.index})

# --- Price ---
# Flag ranges where min > max (parse error) or mid-price below EUR 50k
bad_price_order = df[
    df["price_min"].notna() & df["price_max"].notna() &
    (df["price_min"] > df["price_max"])
]
bad_price_low = df[df["price_mid"].notna() & (df["price_mid"] < 50_000)]
if len(bad_price_order):
    print(f"PRICE - {len(bad_price_order)} row(s) where price_min > price_max (parse error):")
    print(bad_price_order[["unique_id", "full_address", "price_display", "price_min", "price_max"]].to_string())
    flags.update({i: "price_range_inverted" for i in bad_price_order.index})
if len(bad_price_low):
    print(f"PRICE - {len(bad_price_low)} row(s) with mid-price < EUR 50,000 (implausible):")
    print(bad_price_low[["unique_id", "full_address", "price_mid"]].to_string())
    flags.update({i: "price_too_low" for i in bad_price_low.index})

# --- Beds ---
# Flag anything above 10 as likely a parse error
bad_beds = df[df["beds"].notna() & (df["beds"] > 10)]
if len(bad_beds):
    print(f"BEDS - {len(bad_beds)} row(s) with beds > 10:")
    print(bad_beds[["unique_id", "full_address", "beds"]].to_string())
    flags.update({i: "beds_too_high" for i in bad_beds.index})

df["data_quality_flag"] = pd.Series(flags).reindex(df.index, fill_value="")

if not flags:
    print("No quality issues found.")

print("\nSummary statistics:")
print(df[["beds", "baths", "area_m2", "price_min", "price_max", "price_per_m2"]].describe().round(1).to_string())

In [ ]:
# ===========================================================================
# CELL 9 - EXPORT ALL CSVs
# ===========================================================================
# Four CSVs are exported to your myhomes folder:
#
#  1. myhome_listings_full_{ts}.csv   - every column, every row (master file)
#  2. myhome_by_province_{ts}.csv     - aggregated stats by province
#  3. myhome_by_agent_{ts}.csv        - listing count and avg price by agent
#  4. myhome_quality_flags_{ts}.csv   - rows with data quality issues (if any)
#
# utf-8-sig encoding ensures Excel opens the file correctly and displays
# special characters like the euro sign without garbling them.
# ===========================================================================

ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. Full master file
f1 = OUTPUT_DIR / f"myhome_listings_full_{ts}.csv"
df.to_csv(f1, index=False, encoding="utf-8-sig")
print(f"Full data        : {f1}")

# 2. Province summary with EUR formatting so numbers are readable
f2 = OUTPUT_DIR / f"myhome_by_province_{ts}.csv"
prov = (
    df.groupby(["province", "county_standard"], dropna=False)
    .agg(
        listings      =("unique_id",    "count"),
        poa_count     =("is_poa",       "sum"),
        avg_price_mid =("price_mid",    "mean"),
        med_price_mid =("price_mid",    "median"),
        avg_area_m2   =("area_m2",      "mean"),
        avg_price_pm2 =("price_per_m2", "mean"),
    )
    .round(0)
    .reset_index()
    .sort_values("listings", ascending=False)
)
for col in ["avg_price_mid", "med_price_mid", "avg_price_pm2"]:
    prov[col] = prov[col].apply(
        lambda x: f"EUR {int(x):,}" if pd.notna(x) else ""
    )
prov["avg_area_m2"] = prov["avg_area_m2"].apply(
    lambda x: f"{x:.0f} m2" if pd.notna(x) else ""
)
prov.to_csv(f2, index=False, encoding="utf-8-sig")
print(f"Province summary : {f2}")

# 3. Agent summary with EUR formatting
f3 = OUTPUT_DIR / f"myhome_by_agent_{ts}.csv"
agent = (
    df.groupby("agent_name", dropna=False)
    .agg(
        listings      =("unique_id",    "count"),
        avg_price_mid =("price_mid",    "mean"),
        med_price_mid =("price_mid",    "median"),
        poa_count     =("is_poa",       "sum"),
    )
    .round(0)
    .reset_index()
    .sort_values("listings", ascending=False)
)
for col in ["avg_price_mid", "med_price_mid"]:
    agent[col] = agent[col].apply(
        lambda x: f"EUR {int(x):,}" if pd.notna(x) else ""
    )
agent.to_csv(f3, index=False, encoding="utf-8-sig")
print(f"Agent summary    : {f3}")

# 4. Quality flags (only exported if issues were found)
flagged = df[df["data_quality_flag"] != ""]
if len(flagged):
    f4 = OUTPUT_DIR / f"myhome_quality_flags_{ts}.csv"
    flagged[
        ["unique_id", "myhome_id", "full_address", "area_m2",
         "price_display", "price_min", "price_max", "beds", "data_quality_flag"]
    ].to_csv(f4, index=False, encoding="utf-8-sig")
    print(f"Quality flags    : {f4}  ({len(flagged)} rows need review)")
else:
    print("No quality flags - skipping flags CSV.")

print("\nAll exports complete.")

In [ ]:
# ===========================================================================
# CELL 10 - ANALYSIS: PRICE BY PROVINCE AND COUNTY
# ===========================================================================
# Uses price_mid (midpoint of the price range) for all calculations.
# POA listings are excluded from price analysis as they have no value.
# ===========================================================================

priced = df[df["is_poa"] == False]

print("PRICE ANALYSIS BY PROVINCE")
print("=" * 60)
province_price = (
    priced.groupby("province")["price_mid"]
    .agg(
        listings      ="count",
        min_price     ="min",
        max_price     ="max",
        median_price  ="median",
        avg_price     ="mean",
    )
    .round(0)
    .reset_index()
    .sort_values("median_price", ascending=False)
)
print(province_price.to_string(index=False))

print("\n\nPRICE PER M2 BY PROVINCE (priced listings with known area only)")
print("=" * 60)
ppm2 = (
    df[df["price_per_m2"].notna()]
    .groupby("province")["price_per_m2"]
    .agg(
        count              ="count",
        min_price_per_m2   ="min",
        max_price_per_m2   ="max",
        median_price_per_m2="median",
        avg_price_per_m2   ="mean",
    )
    .round(0)
    .reset_index()
    .sort_values("avg_price_per_m2", ascending=False)
)
print(ppm2.to_string(index=False))

print("\n\nPRICE ANALYSIS BY COUNTY (top 10 by listing count)")
print("=" * 60)
county_price = (
    priced.groupby("county_standard")["price_mid"]
    .agg(
        listings     ="count",
        min_price    ="min",
        max_price    ="max",
        median_price ="median",
        avg_price    ="mean",
    )
    .round(0)
    .reset_index()
    .sort_values("listings", ascending=False)
    .head(10)
)
print(county_price.to_string(index=False))

In [ ]:
# ===========================================================================
# CELL 11 - ANALYSIS: MARKET OVERVIEW AND TOP AGENTS
# ===========================================================================
# A summary dashboard covering the full dataset - totals, top agents,
# and property type breakdown.
# ===========================================================================

print("MYHOME.IE NEW HOMES - MARKET OVERVIEW")
print("=" * 60)
print(f"Analysis date          : {datetime.now().strftime('%d %B %Y')}")
print(f"Total listings         : {len(df)}")
print(f"POA listings           : {df['is_poa'].sum()} ({df['is_poa'].mean()*100:.1f}%)")
print(f"Listings with area     : {df['area_m2'].notna().sum()}")
print(f"Listings with beds     : {df['beds'].notna().sum()}")
print()

priced = df[df["is_poa"] == False]
print(f"Priced listings        : {len(priced)}")
if len(priced):
    print(f"  Average mid-price    : EUR {int(priced['price_mid'].mean()):,}")
    print(f"  Median mid-price     : EUR {int(priced['price_mid'].median()):,}")
    print(f"  Cheapest listing     : EUR {int(priced['price_min'].min()):,}")
    print(f"  Most expensive       : EUR {int(priced['price_max'].max()):,}")

print()
print("TOP 10 AGENTS BY LISTING COUNT")
print("=" * 60)
top_agents = (
    df.groupby("agent_name", dropna=False)
    .agg(
        listings =("unique_id", "count"),
        poa      =("is_poa",    "sum"),
    )
    .sort_values("listings", ascending=False)
    .head(10)
)
print(top_agents.to_string())

print()
print("PROPERTY TYPE BREAKDOWN")
print("=" * 60)
print(
    df["property_type"]
    .fillna("Not specified")
    .value_counts()
    .head(15)
    .to_string()
)

print()
print("BEDS BREAKDOWN")
print("=" * 60)
print(
    df["beds"]
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .to_string()
)